# MEGABYTE — Segunda entrega (React e-commerce)

Este notebook explica **dónde está y cómo funciona** cada criterio de la segunda entrega dentro del proyecto `proyectoecommerce`.

> Los fragmentos de código son **JavaScript/React** (se muestran como texto). La única celda ejecutable es una prueba en Python de la API del criterio 7.

## Estructura del proyecto

```
src/
├── App.js                      ← estado principal (useState, LocalStorage, filtros, historial)
├── App.css                     ← estilos del primer corte (sin cambios)
├── SegundaEntrega.css          ← NUEVO: estilos de los componentes nuevos
├── assets/
│   └── banner-megabyte.svg     ← NUEVO: imagen local
├── components/
│   ├── StoreLayout.js          ← layout + Header, ProductCard, Carrito, Checkout (primer corte)
│   ├── HeroBanner.js           ← NUEVO: banner con imagen importada
│   ├── MenuNavegacion.js       ← NUEVO: menú con estado
│   ├── FiltrosAvanzados.js     ← NUEVO: checkbox, radio, range
│   ├── CarritoDropZone.js      ← NUEVO: zona para soltar (Drag and Drop)
│   ├── ConversorDolares.js     ← NUEVO: useEffect + fetch
│   ├── Opiniones.js            ← NUEVO: formulario completo + LocalStorage
│   └── HistorialCompras.js     ← NUEVO: compras guardadas
├── hooks/
│   └── useLocalStorage.js      ← NUEVO: hook personalizado (useState + useEffect + LocalStorage)
├── services/
│   └── api.js                  ← NUEVO: petición fetch a un servidor web
├── data/productos.js           ← catálogo (sin cambios)
└── utils/                      ← validaciones y PDF (sin cambios)
```

## Mapa rápido: criterio → archivo

| # | Criterio | Archivo(s) |
|---|---|---|
| 1 | Todo lo del primer corte | `StoreLayout.js`, `utils/validaciones.js`, `utils/generarFacturaPdf.js` |
| 2 | Formularios y controles con estado | `MenuNavegacion.js`, `FiltrosAvanzados.js`, `Opiniones.js` |
| 3 | Componentes y comunicación | `App.js` → `StoreLayout.js` → componentes hijos (props y callbacks) |
| 4 | Eventos | `onClick`, `onChange`, `onSubmit`, `onDragStart`, `onDragOver`, `onDrop` |
| 5 | `useState` | Todos los componentes nuevos y `App.js` |
| 6 | `useEffect` | `App.js` (título), `ConversorDolares.js`, `hooks/useLocalStorage.js` |
| 7 | `fetch` | `services/api.js` + `ConversorDolares.js` |
| 8 | Propiedad `key` | Listas en Filtros, Menú, Opiniones, Historial |
| 9 | Imágenes | `assets/banner-megabyte.svg` + `HeroBanner.js` |
| 10 | Filtros | `FiltrosAvanzados.js` (vista) + `App.js` (lógica) |
| 11 | Hosting | Carpeta `build/` → Netlify o Vercel |
| 12 | Drag and Drop | `ProductCard` en `StoreLayout.js` + `CarritoDropZone.js` |
| 13 | LocalStorage | `hooks/useLocalStorage.js`, usado en `App.js` y `Opiniones.js` |
| 14 | Valor agregado | Total en USD en vivo, historial de compras, opiniones con estrellas, drag and drop |

## 1) Todo lo del primer corte

Se conserva sin cambios: página principal, enlaces a las especificaciones oficiales, formulario de facturación con validaciones (`utils/validaciones.js`), eventos, objetos (`producto`, `orden`), componentes y `useState`.

Únicos ajustes a archivos existentes (solo se **agregó**, no se borró nada):
- `App.js`: el carrito y los productos ahora se guardan en LocalStorage; se agregan filtros nuevos e historial.
- `StoreLayout.js`: la tarjeta se puede arrastrar y `onFacturaGenerada(orden)` envía la orden para guardarla en el historial.

## 2) Formularios y controles con variables de estado

Un **control controlado** es un input cuyo valor viene de una variable de estado (`value={...}`) y que actualiza ese estado con `onChange`.

### Menú — `src/components/MenuNavegacion.js`
El estado `seccionActiva` vive en `App.js`; el menú lo recibe por props y lo cambia con un callback.

```jsx
<button
  key={opcion.id}
  className={`menu__opcion ${seccionActiva === opcion.id ? "menu__opcion--activa" : ""}`}
  onClick={() => onCambiarSeccion(opcion.id)}
>
  {opcion.etiqueta}
</button>
```

### Casillas de selección, radio y rango — `src/components/FiltrosAvanzados.js`

```jsx
<input type="checkbox"
  checked={filtros.marcas.includes(marca)}
  onChange={() => alternarMarca(marca)} />

<input type="range" min={0} max={precioTope} step={50000}
  value={filtros.precioMaximo}
  onChange={(evento) => actualizar("precioMaximo", Number(evento.target.value))} />

<input type="radio" name="orden" value={opcion.valor}
  checked={filtros.orden === opcion.valor}
  onChange={(evento) => actualizar("orden", evento.target.value)} />
```

### Formulario completo — `src/components/Opiniones.js`
Tiene **caja de texto, select, radio, textarea y checkbox**, todos controlados por un solo objeto de estado:

```jsx
const [formulario, setFormulario] = useState(OPINION_INICIAL);

const manejarCambio = (evento) => {
  const { name, value, type, checked } = evento.target;
  setFormulario({ ...formulario, [name]: type === "checkbox" ? checked : value });
};

<textarea name="comentario" maxLength={MAX_CARACTERES}
  value={formulario.comentario} onChange={manejarCambio} />
<span>{formulario.comentario.length}/{MAX_CARACTERES} caracteres</span>
```

**Para explicar:** una sola función `manejarCambio` sirve para todos los campos porque usa el atributo `name` como clave del objeto. En los checkbox se lee `checked` en vez de `value`.

## 3) Componentes y comunicación entre ellos

```
App.js  (estado: productos, carrito, filtros, historial, seccionActiva)
  └── StoreLayout.js
        ├── Header
        ├── HeroBanner
        ├── MenuNavegacion      ← recibe seccionActiva, avisa con onCambiarSeccion
        ├── FiltrosAvanzados    ← recibe filtros, avisa con onFiltrosChange
        ├── ProductGrid → ProductCard
        ├── Opiniones / HistorialCompras
        ├── CarritoDropZone → CartSidebar   (children)
        ├── ConversorDolares    ← recibe totalCop
        └── CheckoutForm        ← avisa con onFacturaGenerada(orden)
```

- **Padre → hijo:** por **props** (datos). Ej: `<ConversorDolares totalCop={total} />`
- **Hijo → padre:** por **funciones callback**. Ej: `onFiltrosChange({ ...filtros, orden: "precio-asc" })`
- **Composición con `children`:** `CarritoDropZone` envuelve al carrito sin saber qué contiene:

```jsx
<CarritoDropZone onSoltarProducto={onSoltarProducto}>
  <CartSidebar carrito={carrito} onQuitar={onQuitar} total={total} />
</CarritoDropZone>
```

## 4) Eventos generados por un componente

| Evento | Dónde | Qué hace |
|---|---|---|
| `onClick` | `MenuNavegacion.js`, `FiltrosAvanzados.js` (Limpiar), `ConversorDolares.js` (Actualizar) | Cambia sección, limpia filtros, vuelve a consultar la API |
| `onChange` | Filtros y Opiniones | Actualiza el estado del control |
| `onSubmit` | `Opiniones.js` | Valida y guarda la opinión (`evento.preventDefault()` evita recargar la página) |
| `onDragStart` | `ProductCard` en `StoreLayout.js` | Guarda el id del producto que se arrastra |
| `onDragOver` / `onDragLeave` / `onDrop` | `CarritoDropZone.js` | Resalta la zona y agrega el producto soltado |

Ejemplo de evento que **sube** información al padre: `CheckoutForm` llama `onFacturaGenerada(orden)` y `App.js` usa esa orden para guardarla en el historial.

## 5) Hook `useState()`

`useState` devuelve `[valor, funcionParaCambiarlo]`. Cuando se cambia el valor, React vuelve a dibujar el componente.

En `src/App.js`:
```jsx
const [filtros, setFiltros] = useState(FILTROS_INICIALES);
const [seccionActiva, setSeccionActiva] = useState("catalogo");
```

En `src/components/Opiniones.js`:
```jsx
const [formulario, setFormulario] = useState(OPINION_INICIAL);
const [error, setError] = useState("");
const [filtroEstrellas, setFiltroEstrellas] = useState("todas");
```

En `src/components/CarritoDropZone.js` un estado booleano controla el resaltado:
```jsx
const [arrastrandoEncima, setArrastrandoEncima] = useState(false);
```

**Buena práctica usada:** actualizar con la forma funcional cuando el nuevo valor depende del anterior: `setRecargas((actual) => actual + 1)`.

## 6) Hook de efecto `useEffect()`

`useEffect(funcion, [dependencias])` ejecuta código **después** de dibujar, y lo repite solo cuando cambian las dependencias.

### a) Título de la pestaña — `src/App.js`
```jsx
useEffect(() => {
  document.title =
    carrito.length > 0 ? `(${carrito.length}) MEGABYTE - Carrito` : "MEGABYTE - Hardware gamer";
}, [carrito.length]);
```

### b) Petición al servidor con limpieza — `src/components/ConversorDolares.js`
```jsx
useEffect(() => {
  const controlador = new AbortController();
  obtenerTasaUsdCop(controlador.signal)
    .then((resultado) => { setTasa(resultado.tasa); setActualizado(resultado.actualizado); })
    .catch((err) => { if (err.name !== "AbortError") setError("No se pudo consultar..."); });

  return () => controlador.abort();   // limpieza: cancela si el componente se desmonta
}, [recargas]);                        // se repite al pulsar "Actualizar"
```

### c) Guardar en LocalStorage — `src/hooks/useLocalStorage.js`
```jsx
useEffect(() => {
  localStorage.setItem(clave, JSON.stringify(valor));
}, [clave, valor]);
```

| Dependencias | Cuándo se ejecuta |
|---|---|
| `[]` | Solo una vez, al montar |
| `[x]` | Al montar y cada vez que cambia `x` |
| sin arreglo | En cada render (se evita) |

## 7) Peticiones a un servidor web con `fetch`

Archivo: `src/services/api.js`. Se consulta una API pública y gratuita de tasas de cambio para mostrar el total del carrito en **dólares**.

```js
const URL_TASA_CAMBIO = "https://open.er-api.com/v6/latest/USD";

export async function obtenerTasaUsdCop(signal) {
  const respuesta = await fetch(URL_TASA_CAMBIO, { signal });
  if (!respuesta.ok) throw new Error(`Error HTTP ${respuesta.status}`);
  const datos = await respuesta.json();
  return { tasa: datos.rates.COP, actualizado: datos.time_last_update_utc };
}
```

**Pasos para explicar:**
1. `fetch(url)` hace una petición HTTP GET y devuelve una *promesa*.
2. `await` espera la respuesta; `respuesta.ok` indica si el código HTTP es 200-299.
3. `respuesta.json()` convierte el cuerpo de la respuesta a un objeto de JavaScript.
4. El componente `ConversorDolares.js` muestra 3 estados: **cargando**, **error** o **resultado**.

La siguiente celda (Python) hace **la misma petición** para ver qué responde el servidor:

In [ ]:
# Misma petición que hace fetch en src/services/api.js, pero desde Python
import json
import urllib.request

URL = "https://open.er-api.com/v6/latest/USD"
with urllib.request.urlopen(URL) as respuesta:
    datos = json.load(respuesta)

tasa = datos["rates"]["COP"]
print("Resultado:", datos["result"])
print("Actualizado:", datos["time_last_update_utc"])
print(f"1 USD = {tasa:,.2f} COP")

total_carrito_cop = 1_900_000  # ej: una RTX 4060 Ti
print(f"Total ejemplo: ${total_carrito_cop:,} COP ≈ US$ {total_carrito_cop / tasa:,.2f}")

## 8) Propiedad `key` en listas de datos

Cuando se dibuja una lista con `.map()`, React necesita una `key` **única y estable** para saber qué elemento cambió, se agregó o se eliminó.

| Archivo | Lista | key |
|---|---|---|
| `MenuNavegacion.js` | opciones del menú | `opcion.id` |
| `FiltrosAvanzados.js` | marcas / opciones de orden | `marca` / `opcion.valor` |
| `Opiniones.js` | productos del select / opiniones | `producto.id` / `opinion.id` |
| `HistorialCompras.js` | compras / productos de cada compra | `compra.id` / `` `${compra.id}-${indice}` `` |
| `StoreLayout.js` (primer corte) | tarjetas de producto | `producto.id` |

```jsx
{opinionesVisibles.map((opinion) => (
  <li key={opinion.id} className="opiniones__item"> ... </li>
))}
```

**Para explicar:** `opinion.id` se crea con `Date.now()`, así que es único. Usar el índice como key solo es aceptable cuando la lista no se reordena (como los productos dentro de una compra ya hecha).

## 9) Imágenes

Hay dos formas de usar imágenes en el proyecto:

**a) Imagen local importada** — `src/assets/banner-megabyte.svg`, usada en `src/components/HeroBanner.js`:
```jsx
import bannerMegabyte from "../assets/banner-megabyte.svg";

<img className="hero__imagen" src={bannerMegabyte} alt="Ilustración de un chip MEGABYTE" />
```
Al importarla, webpack la incluye en el `build` y le da una ruta final.

**b) Imágenes por URL** (primer corte) — cada producto en `data/productos.js` tiene `imagen`, usada en `ProductCard`:
```jsx
<img src={producto.imagen} alt={producto.nombre} loading="lazy" />
```

**Buenas prácticas:** atributo `alt` para accesibilidad y `loading="lazy"` para cargar las imágenes solo cuando se ven.

## 10) Filtros

**Vista:** `src/components/FiltrosAvanzados.js` · **Lógica:** `productosFiltrados` en `src/App.js`.

Filtros disponibles: texto (buscador), categoría, **marcas**, **precio máximo**, **solo con stock** y **ordenamiento**.

```jsx
const productosFiltrados = useMemo(() => {
  const textoBusqueda = busqueda.trim().toLowerCase();
  const resultado = productos.filter((producto) => {
    const coincideTexto = producto.nombre.toLowerCase().includes(textoBusqueda);
    const coincideCategoria = categoriaFiltro === "Todas" || producto.categoria === categoriaFiltro;
    const coincideMarca = filtros.marcas.length === 0 || filtros.marcas.includes(producto.marca);
    const coincidePrecio = producto.precio <= filtros.precioMaximo;
    const coincideStock = !filtros.soloDisponibles || producto.stock > 0;
    return coincideTexto && coincideCategoria && coincideMarca && coincidePrecio && coincideStock;
  });
  return ordenarProductos(resultado, filtros.orden);
}, [productos, busqueda, categoriaFiltro, filtros]);
```

**Para explicar:**
- `.filter()` deja solo los productos que cumplen **todas** las condiciones.
- `ordenarProductos` hace una copia `[...lista]` antes de `.sort()` para **no mutar** el estado.
- `useMemo` evita recalcular el filtro si nada cambió.
- Extra: en `Opiniones.js` también se filtran las opiniones por número de estrellas.

## 11) Subir la aplicación a un hosting

`npm run build` genera la carpeta **`build/`** con HTML, CSS y JS optimizados (ya está generada).

### Opción A — Netlify Drop (la más rápida)
1. Ejecutar `npm run build`.
2. Entrar a **https://app.netlify.com/drop** e iniciar sesión.
3. Arrastrar la carpeta `build` a la página.
4. Netlify entrega una URL pública (se puede renombrar en *Site settings*).

### Opción B — Vercel con GitHub
1. Subir el proyecto a un repositorio de GitHub.
2. En **vercel.com** → *Add New Project* → importar el repositorio.
3. Vercel detecta Create React App (build: `npm run build`, salida: `build`) y publica. Cada `git push` vuelve a publicar.

**URL publicada:** _(pegar aquí el enlace)_

## 12) Función Drag and Drop

Se usa la API nativa de HTML5. El usuario arrastra una tarjeta de producto y la suelta sobre el carrito.

**Paso 1 — Arrastrar** (`ProductCard` en `src/components/StoreLayout.js`):
```jsx
const manejarDragStart = (evento) => {
  evento.dataTransfer.setData("text/plain", String(producto.id));
  evento.dataTransfer.effectAllowed = "copy";
};

<article draggable={!sinStock} onDragStart={manejarDragStart}> ... </article>
```

**Paso 2 — Soltar** (`src/components/CarritoDropZone.js`):
```jsx
const manejarDragOver = (evento) => {
  evento.preventDefault();          // sin esto el navegador NO permite soltar
  if (!arrastrandoEncima) setArrastrandoEncima(true);
};

const manejarDrop = (evento) => {
  evento.preventDefault();
  setArrastrandoEncima(false);
  const idProducto = Number(evento.dataTransfer.getData("text/plain"));
  if (idProducto) onSoltarProducto(idProducto);
};
```

**Paso 3 — Agregar** (`agregarPorId` en `src/App.js`): busca el producto por id y reutiliza `agregarAlCarrito`, así **se respeta el control de stock** del primer corte.

Los productos agotados tienen `draggable={false}`. Funciona con mouse; en celular se usa el botón *Añadir al carrito*.

## 13) Almacenamiento en LocalStorage

Se creó un **hook personalizado** en `src/hooks/useLocalStorage.js` que funciona igual que `useState` pero guarda el valor en el navegador:

```jsx
export function useLocalStorage(clave, valorInicial) {
  const [valor, setValor] = useState(() => {
    const guardado = localStorage.getItem(clave);
    return guardado !== null ? JSON.parse(guardado) : valorInicial;   // leer
  });

  useEffect(() => {
    localStorage.setItem(clave, JSON.stringify(valor));              // guardar
  }, [clave, valor]);

  return [valor, setValor];
}
```

Uso:

| Clave en LocalStorage | Archivo | Qué guarda |
|---|---|---|
| `megabyte-carrito` | `App.js` | Productos en el carrito |
| `megabyte-productos` | `App.js` | Catálogo con el stock actualizado después de cada compra |
| `megabyte-historial` | `App.js` | Compras realizadas |
| `megabyte-opiniones` | `Opiniones.js` | Opiniones publicadas |

**Para explicar:**
- LocalStorage solo guarda **texto**, por eso se usa `JSON.stringify` al guardar y `JSON.parse` al leer.
- **Para demostrarlo:** agregar productos al carrito → recargar la página (F5) → el carrito sigue ahí.
- Se ve en el navegador: **F12 → Application → Local Storage**. Borrando esas claves la tienda vuelve a su estado inicial.

## 14) Valor agregado

| Funcionalidad | Archivo | Por qué aporta |
|---|---|---|
| Total del carrito en **dólares** con tasa en vivo | `ConversorDolares.js`, `services/api.js` | El cliente compara con precios internacionales de hardware |
| **Historial de compras** ("Mis compras") | `HistorialCompras.js`, `App.js` | Las facturas generadas quedan guardadas aunque se cierre el navegador |
| **Opiniones** con estrellas, contador de caracteres y filtro | `Opiniones.js` | Confianza para otros compradores |
| **Drag and Drop** al carrito | `CarritoDropZone.js` | Forma rápida e interactiva de comprar |
| **Hook personalizado** `useLocalStorage` | `hooks/useLocalStorage.js` | Código reutilizable (buena práctica) |
| Contador del carrito en la **pestaña del navegador** | `App.js` (useEffect) | El usuario ve cuántos productos lleva |
| Stock actualizado que **persiste** | `App.js` | Tras comprar, el inventario baja y se mantiene |

## Guion sugerido para la demo

1. **Inicio:** mostrar el banner (imagen local) y el menú.
2. **Filtros:** marcar *NVIDIA*, bajar el precio máximo, ordenar por *Menor precio*, marcar *Solo con stock* → *Limpiar filtros*.
3. **Drag and Drop:** arrastrar una tarjeta al carrito (se resalta en azul).
4. **fetch:** ver el total en USD y pulsar *Actualizar*.
5. **LocalStorage:** recargar con F5 → el carrito sigue. Mostrar F12 → Application → Local Storage.
6. **Compra:** llenar la facturación → descarga del PDF → el stock baja → menú *Mis compras* muestra la compra.
7. **Opiniones:** dejar una opinión (texto, select, estrellas con radio, textarea, checkbox) y filtrarla por estrellas.
8. **Hosting:** abrir la URL publicada.

## Vocabulario técnico clave

- **Componente:** función que devuelve JSX y se reutiliza.
- **Props:** datos que el padre le pasa al hijo (solo lectura).
- **Estado (state):** datos que cambian y hacen que React vuelva a dibujar.
- **Callback:** función que el padre pasa al hijo para que este le avise algo.
- **Hook:** función especial de React que empieza con `use` (`useState`, `useEffect`, `useMemo`, `useLocalStorage`).
- **Efecto secundario:** acción fuera del render (petición HTTP, LocalStorage, `document.title`).
- **Promesa / async-await:** forma de trabajar con operaciones que tardan, como `fetch`.
- **Inmutabilidad:** no modificar el estado directamente; crear copias (`[...lista]`, `{ ...objeto }`).